# TasteTwin Model Training (Colab Optimized)
This notebook will download the Food-101 dataset and train the 3 models using a free Google GPU.

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn tqdm

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import os
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cpu':
    print("\nWARNING: You are using a CPU! Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

In [ ]:
# 1. Download and Prepare Data
data_dir = './data'
os.makedirs(data_dir, exist_ok=True)

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Downloading Food-101 Dataset... This takes about 2 minutes.")
train_dataset = torchvision.datasets.Food101(root=data_dir, split='train', download=True, transform=transform_train)
val_dataset = torchvision.datasets.Food101(root=data_dir, split='test', download=True, transform=transform_val)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
class_names = train_dataset.classes
print(f"Dataset ready. {len(class_names)} classes.")

In [ ]:
# 2. Define Training Loop Function
def train_model(model_name, epochs=15):
    print(f"\n--- Training {model_name.upper()} ---")
    
    if model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, len(class_names))
    elif model_name == 'efficientnet':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
    elif model_name == 'vit':
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads.head = nn.Linear(model.heads.head.in_features, len(class_names))
        
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    metrics = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        model.train()
        running_loss = 0.0
        
        for inputs, labels in tqdm(train_loader, desc="Training"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            
        epoch_train_loss = running_loss / len(train_dataset)
        metrics['train_loss'].append(epoch_train_loss)
        
        model.eval()
        running_loss = 0.0
        correct = 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validating"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                correct += torch.sum(preds == labels.data)
                
        epoch_val_loss = running_loss / len(val_dataset)
        epoch_val_acc = correct.double().item() / len(val_dataset)
        
        metrics['val_loss'].append(epoch_val_loss)
        metrics['val_acc'].append(epoch_val_acc)
        
        print(f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
        
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(model.state_dict(), f'{model_name}_best.pth')
            print(f"-> Saved new best model!")
            
    with open(f'{model_name}_metrics.json', 'w') as f:
        json.dump(metrics, f)
    print(f"\nCompleted {model_name}. Best Acc: {best_acc:.4f}")

In [ ]:
# 3. Run the Training! (These cells can run back to back)
train_model('efficientnet', epochs=15)

In [ ]:
train_model('resnet50', epochs=15)

In [ ]:
train_model('vit', epochs=15)

# Download the Files to your Computer!
Run the cell below to download your newly trained models and metrics back to your own computer so we can put them in the TasteTwin/backend folder.

In [ ]:
from google.colab import files
import glob
import os

# Download all .pth and .json files
files_to_download = glob.glob('*.pth') + glob.glob('*.json')
for f in files_to_download:
    print(f"Downloading {f}...")
    files.download(f)